# Classification Project: Online Shoppers Purchasing Intention
## Notebook 2: Nested Cross-Validation and Data-Leakage-Proof Pipelines

In this phase, we build the core machine learning architecture. Following the strict methodological rule **"Never do preprocessing before splitting D"**, we will encapsulate all data transformations—including Outlier Removal, Feature Scaling, and SMOTE—inside an `imblearn` Pipeline.

### Objectives:
1. **Hold-out Split:** Isolate 20% of the data as a completely untouched Test Set for the final evaluation (Notebook 3).
2. **Custom LOF Sampler:** Engineer a custom pipeline component to remove multivariate outliers dynamically only during the training phase of the Cross-Validation.
3. **Nested Cross-Validation:** Perform an unbiased comparison between a baseline model (**Random Forest**) and an advanced gradient boosting algorithm (**XGBoost**).
4. **Final Model Selection:** Train the winning architecture on the full training set and save it for Explainable AI (XAI) analysis.

> Why is this not Data Leakage? Encoding strings to numbers does not involve calculating any dataset-wide statistics (like means or variances). It is purely a format change, so it is safe to do before splitting the data.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split

print("--- 1. DATA LOADING & ENCODING ---")
current_dir = os.path.dirname(os.path.abspath('__file__'))
file_path = os.path.join(current_dir, 'online_shoppers_intention.csv')

df = pd.read_csv(file_path).dropna().reset_index(drop=True)

# Basic Encoding (No statistical leakage here, just format conversion)
df['Weekend'] = df['Weekend'].astype(int)
df['Revenue'] = df['Revenue'].astype(int)
# One-Hot Encoding for the few string categorical variables
df = pd.get_dummies(df, columns=['Month', 'VisitorType'], drop_first=True)

X = df.drop(columns=['Revenue'])
y = df['Revenue']

print("--- 2. STRICT HOLD-OUT SPLIT (D_train vs D_test) ---")
# We isolate 20% of the data. This set will NOT be seen by the Cross-Validation.
# Stratify ensures the 85/15 class imbalance is maintained in both splits.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print(f"Training Set (D_train): {X_train.shape[0]} samples")
print(f"Test Set (D_test): {X_test.shape[0]} samples (LOCKED AWAY)")

print("--- 3. SAVING TEST SET TO DEDICATED DIRECTORY ---")
# Define the directory name
output_dir = os.path.join(current_dir, '2_locked_test_data')

# Create the directory if it does not exist (prevents errors on multiple runs)
os.makedirs(output_dir, exist_ok=True)

# Define the full file paths
x_test_path = os.path.join(output_dir, 'X_test_locked.csv')
y_test_path = os.path.join(output_dir, 'y_test_locked.csv')

# Save the test set to the new directory for Notebook 3
X_test.to_csv(x_test_path, index=False)
y_test.to_csv(y_test_path, index=False)

print(f"Test data successfully saved inside: {output_dir}")

--- 1. DATA LOADING & ENCODING ---
--- 2. STRICT HOLD-OUT SPLIT (D_train vs D_test) ---
Training Set (D_train): 9864 samples
Test Set (D_test): 2466 samples (LOCKED AWAY)
--- 3. SAVING TEST SET TO DEDICATED DIRECTORY ---
Test data successfully saved inside: c:\Users\paris\OneDrive - University of Pisa\Desktop\progetto_mining\Data-Mining-ML-project\2_locked_test_data


### 2. Engineering the Custom LOF Sampler
Scikit-Learn's `LocalOutlierFactor` is not natively designed to work inside a classification pipeline (it lacks a standard `transform` method). To respect the *"Zero Leakage"* rule, we must prevent the model from identifying outliers using test-fold data.

We solve this by creating a custom class that inherits from `BaseEstimator` and `SamplerMixin`. This tricks the `imblearn` pipeline into treating LOF as an under-sampling technique.
* **During `fit` (Training Fold):** It calculates densities, flags outliers, and removes them from the training data.
* **During `predict` (Validation Fold):** The pipeline automatically skips the sampler, forcing the model to predict on real-world, uncleaned data (including natural outliers).


In [ ]:
from imblearn.base import BaseSampler
from sklearn.neighbors import LocalOutlierFactor
import pandas as pd

class LOF_Sampler(BaseSampler):
    """
    Custom Imblearn Sampler that uses Local Outlier Factor to remove anomalies 
    ONLY from the training folds during Cross-Validation.
    """
    
    # REQUIRED BY IMBLEARN: Defines how the pipeline should handle this step
    _sampling_type = 'clean-sampling'
    
    # REQUIRED BY SCIKIT-LEARN 1.2+: Explicit parameter type validation
    _parameter_constraints = {
        "n_neighbors": [int],
        "contamination": [float, str],
        "sampling_strategy": [str, type(None)] # Satisfies BaseSampler internals
    }

    def __init__(self, n_neighbors=100, contamination=0.05, sampling_strategy="auto"):
        self.n_neighbors = n_neighbors
        self.contamination = contamination
        self.sampling_strategy = sampling_strategy # Required by BaseSampler
        
    def _fit_resample(self, X, y):
        # Initialize LOF
        lof = LocalOutlierFactor(n_neighbors=self.n_neighbors, contamination=self.contamination)
        
        # fit_predict returns 1 for inliers and -1 for outliers
        outlier_flags = lof.fit_predict(X)
        
        # Create a boolean mask to keep only the normal data
        mask = outlier_flags == 1
        
        # BaseSampler's validation often converts DataFrames to NumPy arrays.
        # This safely slices the data regardless of the format it receives.
        if isinstance(X, pd.DataFrame) or hasattr(X, 'iloc'):
            return X.iloc[mask], y.iloc[mask]
        else:
            return X[mask], y[mask]

print("Custom LOF_Sampler successfully defined and fully compliant with imblearn internals.")

Custom LOF_Sampler successfully defined and fully compliant with imblearn internals.


### 3. Nested Cross-Validation & Pipeline Construction
We now construct the full rigorous pipeline:
`LOF_Sampler` -> `StandardScaler` -> `SMOTE` -> `Classifier`

We will compare two models using a **Nested Cross-Validation** approach:
* **Inner Loop (GridSearchCV):** Finds the optimal hyperparameters (including the SMOTE ratio and Tree depth).
* **Outer Loop (cross_val_score):** Evaluates the unbiased generalization performance of the tuned pipeline.

We optimize for the **Macro F1-Score** to ensure the model balances precision and recall on the minority class.

In [ ]:
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score
import warnings
warnings.filterwarnings('ignore') # Keep the output clean

print("--- PIPELINE INITIALIZATION ---")

# 1. Random Forest Pipeline
rf_pipe = ImbPipeline([
    ('lof', LOF_Sampler(n_neighbors=100, contamination=0.05)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 2. XGBoost Pipeline
xgb_pipe = ImbPipeline([
    ('lof', LOF_Sampler(n_neighbors=100, contamination=0.05)),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# --- PARAMETER GRIDS (Inner Loop) ---
# We keep grids compact to optimize computational time while ensuring robust tuning
rf_grid = {
    'smote__sampling_strategy': [0.7, 1.0], # Ratio of minority to majority class
    'classifier__n_estimators': [100],
    'classifier__max_depth': [5, 10]
}

xgb_grid = {
    'smote__sampling_strategy': [0.7, 1.0],
    'classifier__n_estimators': [100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1]
}

# --- NESTED CROSS-VALIDATION ---
print("\nExecuting Nested Cross-Validation (Outer CV: 5 folds, Inner CV: 3 folds)...")
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average='macro')

# Grid Searches
rf_gs = GridSearchCV(rf_pipe, rf_grid, cv=cv_inner, scoring=scorer, n_jobs=-1)
xgb_gs = GridSearchCV(xgb_pipe, xgb_grid, cv=cv_inner, scoring=scorer, n_jobs=-1)

# Outer Loop Evaluation
rf_scores = cross_val_score(rf_gs, X_train, y_train, cv=cv_outer, scoring=scorer, n_jobs=-1)
print(f"Random Forest -> Macro F1-Score: {rf_scores.mean():.4f} (+/- {rf_scores.std() * 2:.4f})")

xgb_scores = cross_val_score(xgb_gs, X_train, y_train, cv=cv_outer, scoring=scorer, n_jobs=-1)
print(f"XGBoost       -> Macro F1-Score: {xgb_scores.mean():.4f} (+/- {xgb_scores.std() * 2:.4f})")

# Determine Winner
model_scores = {'Random Forest': rf_scores.mean(), 'XGBoost': xgb_scores.mean()}
best_model_name = max(model_scores, key=model_scores.get)
print(f"\nWINNER: {best_model_name}!")

--- PIPELINE INITIALIZATION ---

Executing Nested Cross-Validation (Outer CV: 5 folds, Inner CV: 3 folds)...


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\paris\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\paris\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\paris\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py", line 1053, in fit
    self._run_search(evaluate_candidates)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "

### 4. Training the Final Production Model
Nested CV provides an unbiased *estimate* of our methodology's performance, but it does not return a single deployable model (it trains multiple models across the outer folds). 

Now that we have proven our methodology and selected the winning architecture, we will perform one final standard `GridSearchCV` over the **entire** $D_{train}$ set to find the absolute best hyperparameters. We will then save this final fitted pipeline to disk.

In [ ]:
print(f"--- TRAINING FINAL {best_model_name.upper()} MODEL ON ENTIRE D_TRAIN ---")

# Select the winning pipeline and grid
if best_model_name == 'XGBoost':
    final_gs = GridSearchCV(xgb_pipe, xgb_grid, cv=cv_outer, scoring=scorer, n_jobs=-1)
else:
    final_gs = GridSearchCV(rf_pipe, rf_grid, cv=cv_outer, scoring=scorer, n_jobs=-1)

# Fit on the entire training dataset
final_gs.fit(X_train, y_train)

print(f"Best Hyperparameters found:")
for param, value in final_gs.best_params_.items():
    print(f" - {param}: {value}")

print(f"Best Validation Macro F1-Score: {final_gs.best_score_:.4f}")

# Extract the absolute best pipeline
best_pipeline = final_gs.best_estimator_

# Save the model to disk using joblib
model_filename = 'final_best_pipeline.pkl'
joblib.dump(best_pipeline, model_filename)
print(f"\nFinal model successfully saved as '{model_filename}'")
print("We are now ready to move to Notebook 3 for final Testing, Ablation, and SHAP Explainability!")

--- TRAINING FINAL XGBOOST MODEL ON ENTIRE D_TRAIN ---
Best Hyperparameters found:
 - classifier__learning_rate: 0.05
 - classifier__max_depth: 5
 - classifier__n_estimators: 100
 - smote__sampling_strategy: 0.7
Best Validation Macro F1-Score: 0.8159

Final model successfully saved as 'final_best_pipeline.pkl'
We are now ready to move to Notebook 3 for final Testing, Ablation, and SHAP Explainability!
